## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# API Rate Limiting and Optimization Strategies

When you build an application that communicates with a server, you can't send an unlimited number of requests. Servers impose **rate limits** to ensure stability, prevent abuse, and provide fair usage for all users. The KiteConnect API is no different.

Ignoring these limits will lead to your requests being blocked (an HTTP 429 error), which can break your application. This notebook will teach you:

1.  **What API rate limits are** and why they are important.
2.  How to **demonstrate hitting the rate limit** (the wrong way to make calls).
3.  How to **optimize API usage** using powerful strategies like **batch processing** and **throttling**.

## Step 1: Authentication

First, let's get our authenticated `kite` object.

In [1]:
import pandas as pd
from kiteconnect import KiteConnect
import time

# This block simulates the presence of your kite_quantra.py file
try:
    from kite_quantra import KiteAuth
except ImportError:
    print("Could not import KiteAuth. Defining it locally for this demo.")
    class KiteAuth:
        def __init__(self, api_key, api_secret):
            self.api_key, self.api_secret, self.kite = api_key, api_secret, None
        def authenticate(self):
            try:
                kite = KiteConnect(api_key=self.api_key)
                print("Login URL: ", kite.login_url())
                request_token = input("Enter request_token: ").strip()
                data = kite.generate_session(request_token, api_secret=self.api_secret)
                kite.set_access_token(data['access_token'])
                self.kite = kite
                print("Authentication Successful!")
                return kite
            except Exception as e:
                print(f"Auth Error: {e}")
                return None

# --- Start Authentication ---
API_KEY = "YOUR_API_KEY"
API_SECRET = "YOUR_API_SECRET"

kite = None
if API_KEY != "YOUR_API_KEY":
    auth = KiteAuth(API_KEY, API_SECRET)
    kite = auth.authenticate()
else:
    print("Please update your API_KEY and API_SECRET to run.")

Please click on the URL below to login:
https://kite.zerodha.com/connect/login?api_key=tj19jzf6vgdl162v&v=3
Enter the request token from redirect URL: xl275AOfMfddtmbqPR1tOjEQxbTMONkH
Authentication successful!
Welcome, Varun Kumar Pothula!


## Understanding KiteConnect Rate Limits

The KiteConnect API has a general rate limit of **10 requests per second**. If you exceed this, the server will respond with an error, and the `kiteconnect` library will raise a `kiteconnect.exceptions.TokenException` (as repeated rapid requests are often a sign of an issue with the session token or its usage).

Let's see this in action.

### The Wrong Way: Hitting the Limit

The code below attempts to fetch the LTP for a single instrument 20 times in a rapid loop. This will almost certainly trigger the rate limit.

In [9]:
if kite:
    print("--- Demonstrating Rate Limit Error ---")
    # This is an anti-pattern. DO NOT DO THIS.
    error_count = 0
    for i in range(200):
        try:
            ltp = kite.ltp('NSE:INFY')
            print(f"Request {i+1}: Success - LTP for INFY is {ltp['NSE:INFY']['last_price']}")
        except Exception as e:
            print(f"Request {i+1}: FAILED - {e.__class__.__name__}")
            error_count += 1
            
    if error_count > 0:
        print(f"\nAs expected, we hit the rate limit and got {error_count} errors.")
else:
    print("Kite object not authenticated.")

--- Demonstrating Rate Limit Error ---
Request 1: Success - LTP for INFY is 1599.2
Request 2: Success - LTP for INFY is 1599.2
Request 3: Success - LTP for INFY is 1599.2
Request 4: Success - LTP for INFY is 1599.2
Request 5: Success - LTP for INFY is 1599.2
Request 6: Success - LTP for INFY is 1599.2
Request 7: Success - LTP for INFY is 1599.2
Request 8: Success - LTP for INFY is 1599.2
Request 9: Success - LTP for INFY is 1599.2
Request 10: Success - LTP for INFY is 1599.2
Request 11: Success - LTP for INFY is 1599.2
Request 12: Success - LTP for INFY is 1599.2
Request 13: Success - LTP for INFY is 1599.2
Request 14: Success - LTP for INFY is 1599.2
Request 15: Success - LTP for INFY is 1599.2
Request 16: Success - LTP for INFY is 1599.2
Request 17: Success - LTP for INFY is 1599.2
Request 18: Success - LTP for INFY is 1599.2
Request 19: Success - LTP for INFY is 1599.8
Request 20: Success - LTP for INFY is 1599.8
Request 21: Success - LTP for INFY is 1599.8
Request 22: Success - LTP

## Optimization Strategy 1: Batch Processing

The most important optimization strategy is **batching**. Instead of making one API call per instrument, you can request data for multiple instruments in a single call. Most of the quote functions (`ltp`, `ohlc`, `quote`) support this.

Let's say we want the LTP for 50 instruments. Instead of 50 API calls, we make just **one**.

In [4]:
# Let's get a list of 50 instruments from the NSE segment for our test
instruments_to_fetch = []
if kite:
    try:
        all_instruments = kite.instruments('NSE')
        df = pd.DataFrame(all_instruments)
        # Get 50 liquid, well-known stocks
        instruments_to_fetch = ['NSE:' + s for s in df[df.segment == 'NSE']['tradingsymbol'].head(50).tolist()]
        print(f"Prepared a list of {len(instruments_to_fetch)} instruments.")
    except Exception as e:
        print(f"Could not fetch instrument list: {e}")
        
# Now, let's fetch LTP for all 50 in a single call
if instruments_to_fetch:
    try:
        print("\n--- OPTIMIZED: Using Batch Processing ---")
        start_time = time.time()
        
        # ONE single API call for 50 instruments
        ltp_data = kite.ltp(instruments_to_fetch)
        
        end_time = time.time()
        
        print(f"Successfully fetched LTP for {len(ltp_data)} instruments in {end_time - start_time:.2f} seconds.")
        print("This consumed only 1 API request.")
        # print(pd.DataFrame(ltp_data).T.head())
    except Exception as e:
        print(f"Error during batch fetch: {e}")

Prepared a list of 50 instruments.

--- OPTIMIZED: Using Batch Processing ---
Successfully fetched LTP for 50 instruments in 0.07 seconds.
This consumed only 1 API request.


## Optimization Strategy 2: Throttling Requests

Even with batching, you might need to refresh data periodically. If you need updates faster than the rate limit allows (e.g., multiple times a second), you should use WebSockets (a topic for another notebook). For most other cases, **throttling** is the answer.

Throttling simply means introducing a delay between your API calls. If the limit is 10 requests/second, waiting just 0.1 seconds between requests is enough to stay within the limit. A more conservative approach is to wait a full second or more.

In [5]:
if kite:
    print("--- OPTIMIZED: Using Throttling ---")
    # We will make 5 requests, but wait 1 second between each one.
    for i in range(5):
        try:
            start_time = time.time()
            print(f"Request {i+1} at {time.strftime('%H:%M:%S')}")
            # Fetching a batch in each throttled request is the best of both worlds
            ltp = kite.ltp(['NSE:INFY', 'NSE:RELIANCE'])
            print(f"  > Success. LTP for INFY: {ltp['NSE:INFY']['last_price']}")
            
            # Wait for 1 second before the next iteration
            time.sleep(1)
            
        except Exception as e:
            print(f"Request {i+1}: FAILED - {e}")
            break # Exit loop on failure
else:
    print("Kite object not authenticated.")

--- OPTIMIZED: Using Throttling ---
Request 1 at 09:28:01
  > Success. LTP for INFY: 1594.3
Request 2 at 09:28:02
  > Success. LTP for INFY: 1594.3
Request 3 at 09:28:03
  > Success. LTP for INFY: 1594.2
Request 4 at 09:28:05
  > Success. LTP for INFY: 1594.3
Request 5 at 09:28:06
  > Success. LTP for INFY: 1594


## Conclusion

Efficient API usage is a hallmark of a well-built application. By understanding and respecting rate limits, you ensure your program runs smoothly without interruptions. The two key strategies to remember are:

1.  **Batch Processing**: Always prefer making one call for many instruments over many calls for one instrument. This is the single most effective optimization.
2.  **Throttling**: When you need to poll for updates, introduce a sensible delay (`time.sleep()`) between your requests to stay well within the API rate limits.

Apart from rate limiting, you might get other errors or exceptions raised as well. For more details on errors and exceptions, you can find the link for the documentation <a href='https://kite.trade/docs/connect/v3/exceptions/' target="_blank">here</a>.

Mastering these techniques will allow you to build robust and scalable trading applications with KiteConnect.<br><br>